This notebook tests the textbook implementation of the Jacobian and the kinematic tree forward kinematics.

In [1]:
import numpy as np
from robot import Robot, MotomanRobot, KinematicsNode, KinematicsTree

In [2]:
robot = MotomanRobot()
kin_tree = KinematicsTree(robot.mj_model)

In [3]:
link_name = 'arm_left_link_7_t'
pose = kin_tree.get_link_pose(link_name, True)
print(pose)

node.product_exp_xi_theta: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
node.link_pose_q_zeroed: [[ 1.00000000e+00 -0.00000000e+00  0.00000000e+00  1.00000000e-01]
 [ 0.00000000e+00 -2.22044605e-16  1.00000000e+00  1.14000000e+00]
 [-0.00000000e+00 -1.00000000e+00 -2.22044605e-16  1.20000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e-01]
 [ 0.00000000e+00 -2.22044605e-16  1.00000000e+00  1.14000000e+00]
 [ 0.00000000e+00 -1.00000000e+00 -2.22044605e-16  1.20000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [4]:
# compare the pose with the one from mujoco
print('pose from mujoco:')
pose_mujoco = robot.get_link_pose(link_name)
print(pose_mujoco)

pose from mujoco:
[[ 1.00000000e+00 -0.00000000e+00  0.00000000e+00  1.00000000e-01]
 [ 0.00000000e+00 -2.22044605e-16  1.00000000e+00  1.14000000e+00]
 [-0.00000000e+00 -1.00000000e+00 -2.22044605e-16  1.20000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [5]:
# try for a random joint and compare
q = np.random.uniform(robot.robot_joint_limits[:, 0], robot.robot_joint_limits[:, 1])
robot.set_joint_values(q)
kin_tree.update_joint_values(robot.mj_data)

In [6]:
link_name = 'arm_left_link_7_t'
print('pose from mujoco:')
pose_mujoco = robot.get_link_pose(link_name)
print(pose_mujoco)
pose = kin_tree.get_link_pose(link_name, True)
print(pose)

pose from mujoco:
[[-0.96672242  0.20532209 -0.1526126  -0.59845951]
 [-0.22307279 -0.96858242  0.10993915 -0.21486399]
 [-0.12524494  0.14032436  0.98215212  1.63128678]
 [ 0.          0.          0.          1.        ]]
node.product_exp_xi_theta: [[-0.96672242 -0.1526126  -0.20532209 -0.0814224 ]
 [-0.22307279  0.10993915  0.96858242 -1.48018624]
 [-0.12524494  0.98215212 -0.14032436  0.69254709]
 [ 0.          0.          0.          1.        ]]
node.link_pose_q_zeroed: [[ 1.00000000e+00 -0.00000000e+00  0.00000000e+00  1.00000000e-01]
 [ 0.00000000e+00 -2.22044605e-16  1.00000000e+00  1.14000000e+00]
 [-0.00000000e+00 -1.00000000e+00 -2.22044605e-16  1.20000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[[-0.96672242  0.20532209 -0.1526126  -0.59845951]
 [-0.22307279 -0.96858242  0.10993915 -0.21486399]
 [-0.12524494  0.14032436  0.98215212  1.63128678]
 [ 0.          0.          0.          1.        ]]


check for the jacobian computation

In [7]:
spatial_jacobian = kin_tree.compute_spatial_jacobian_cols(True)
jacs = kin_tree.get_link_spatial_jacobian(link_name, True, False)
# compute the jacobian from mujoco
jacs_mj = robot.get_link_spatial_jacobian(link_name)

In [8]:
# print(jacs)
# print(jacs_mj)
link_idx = kin_tree.mj_model.body(link_name).id
node = kin_tree.nodes[link_idx]
print(node.relavent_joint_indices)
print(len(node.ancestor_nodes))

[7, 0, 1, 2, 3, 4, 5, 6]
9


In [9]:
assert np.allclose(jacs[3:,8:], jacs_mj[3:,8:])
print(jacs[:,:8])
print(jacs_mj[:,:8])
# verified: the rotation part of the jacbian is the same as mujoco, the translation is different

[[ 0.00000000e+00  5.69823851e-01  5.54855313e-01 -5.19481165e-01
  -1.88407045e-01 -1.34832734e+00 -2.69346878e-02  3.90371397e-01]
 [ 0.00000000e+00 -1.05607802e+00  5.49825702e-01 -7.82633131e-01
   1.25810599e+00 -9.01653120e-02  1.54996889e+00 -3.38823366e-01]
 [ 0.00000000e+00  1.00000000e-01  1.47124156e-01 -1.46188654e-01
  -1.79082629e-01 -5.38137811e-01  2.32555977e-01  9.85850798e-02]
 [ 0.00000000e+00 -8.80065014e-01  2.63631614e-01 -7.66391061e-01
   6.01630040e-01  5.05589671e-02  9.88022804e-01  1.52612596e-01]
 [ 0.00000000e+00 -4.74853209e-01 -4.88599330e-01  4.17510948e-01
   2.00168917e-01  9.56608044e-01 -5.96548723e-03 -1.09939148e-01]
 [ 1.00000000e+00  2.22044605e-16  8.31726558e-01  4.88189871e-01
   7.73287592e-01 -2.86957908e-01  1.54192577e-01 -9.82152116e-01]]
[[ 2.77555756e-17  5.69823851e-01  5.54855313e-01 -5.19481165e-01
  -1.88407045e-01 -1.34832734e+00 -2.69346878e-02  3.90371397e-01]
 [ 1.11022302e-16 -1.05607802e+00  5.49825702e-01 -7.82633131e-01
  

now we verify the Jacobian using finite differencing.
according to the textbook, there is 
$$J_{st}^s(\theta)=[\xi_1,\xi_2',\dots,\xi_n']=
[(\frac{\partial g_{st}}{\partial\theta_1}g_{st}^{-1})^V,\dots,(\frac{\partial g_{st}}{\partial\theta_n}g_{st}^{-1})^V]$$
Hence we verify with the analytical Jacobian $\frac{\partial g_{st}}{\partial\theta}$

In [10]:
def naive_finite_diff(func, x):
    # given a function and the point for differentiation, compute the naive finite diff
    # x is a numpy array of shape (n,)
    diff = 1e-10
    grad = np.zeros(x.shape)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += diff
        x_minus = x.copy()
        x_minus[i] -= diff
        grad[i] = (func(x_plus) - func(x_minus)) / (2*diff)
    return grad

def naive_finite_diff_jacobian(func, x, eps=1e-12):
    # given a function and the point for differentiation, compute the naive finite diff
    # func returns array of shape (M,)  #(M can be a list)
    # x is a numpy array of shape (n,)
    # result of shape (M,n)
    jacobian = np.zeros(list(func(x).shape)+[len(x)])
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += eps
        x_minus = x.copy()
        x_minus[i] -= eps
        jacobian[...,i] = (func(x_plus) - func(x_minus)) / (2*eps)
    return jacobian

In [11]:
import so3
import se3

def get_analytical_jacobian(spatial_jac, pose):
    """
    spatial_jac is of shape (6, n)
    """
    jac = se3.hat(spatial_jac.T)  # nx4x4
    jac = jac @ pose
    jac = np.transpose(jac, (1,2,0))  # 4x4xn
    return jac

In [23]:
# try for a random joint and compare
q = np.random.uniform(robot.robot_joint_limits[:, 0], robot.robot_joint_limits[:, 1])
robot.set_joint_values(q)
kin_tree.update_joint_values(robot.mj_data)

link_name = 'arm_left_link_7_t'
print('pose from mujoco:')
pose_mujoco = robot.get_link_pose(link_name)
# print(pose_mujoco)
pose = kin_tree.get_link_pose(link_name, True)
# print(pose)
assert np.allclose(pose, pose_mujoco)

spatial_jacobian = kin_tree.compute_spatial_jacobian_cols(True)
jacs = kin_tree.get_link_spatial_jacobian(link_name, True, False)
# compute the jacobian from mujoco
jacs_mj = robot.get_link_spatial_jacobian_mj(link_name)

assert np.allclose(jacs[3:,:], jacs_mj[3:,:])

pose from mujoco:
node.product_exp_xi_theta: [[ 0.71548831 -0.51874757 -0.46795024  0.54826978]
 [-0.40124432 -0.8534518   0.33260039  0.62114731]
 [-0.57190862 -0.05020932 -0.81877931  1.82838225]
 [ 0.          0.          0.          1.        ]]
node.link_pose_q_zeroed: [[ 1.00000000e+00 -0.00000000e+00  0.00000000e+00  1.00000000e-01]
 [ 0.00000000e+00 -2.22044605e-16  1.00000000e+00  1.14000000e+00]
 [-0.00000000e+00 -1.00000000e+00 -2.22044605e-16  1.20000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [24]:
print(jacs[:3,:8])
print(jacs_mj[:3,:8])

[[ 0.         -0.38446181  0.88363795 -0.7589431   0.64208697 -0.38673124
  -0.42022119 -0.62386774]
 [ 0.         -1.13674497 -0.44236113 -0.88583992 -0.58009309  0.34438978
  -0.75588213  0.40618739]
 [ 0.          0.1         0.22747659 -0.0058991   0.50423561 -0.3018245
  -0.11470318 -0.45870924]]
[[-0.00720829 -0.1501267   0.29258073 -0.35082163  0.29146617  0.0937114
  -0.06008892  0.        ]
 [-0.53309391 -0.44388224  0.03225767 -0.13310576 -0.24834939 -0.06003868
   0.02828409  0.        ]
 [ 0.          0.26396688 -0.2079931   0.28414196  0.24541614  0.05233199
   0.14005117  0.        ]]


In [25]:
analytical_jac = get_analytical_jacobian(jacs, pose)
analytical_jac_mj = get_analytical_jacobian(jacs_mj, pose_mujoco)

In [26]:
q_temp = np.array(q)
def link_pose(q, link_name):
    robot.set_selected_joint_values(q)
    return robot.get_link_pose(link_name)

finite_diff_analytical_jac = naive_finite_diff_jacobian(lambda x: link_pose(x,link_name), q_temp)

In [27]:
assert np.allclose(analytical_jac[:3,:3,:], analytical_jac_mj[:3,:3,:])  # rotation part is the same
diff = analytical_jac - finite_diff_analytical_jac
print(diff[:3,:3,0])

[[-1.01304114e-04  5.32491284e-06 -1.98681560e-04]
 [-1.33691775e-04 -8.76617204e-06  1.70667102e-04]
 [ 0.00000000e+00 -1.11022302e-04  1.63064007e-04]]


In [28]:
print(diff[:3,3,:])

[[-4.73537914e-05 -2.45443903e-05  3.69611357e-05  8.84144524e-06
  -2.28825641e-05 -4.69307721e-05 -2.58573145e-05  1.11022302e-16
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 4.90612294e-05  2.65605785e-05  1.95665251e-05  2.38567739e-05
   1.04645730e-04 -1.72444087e-05 -1.27171869e-05  5.55111512e-17
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  6.68698016e-05  6.26900724e-05 -1.96225828e-05
   5.68503779e-05 -1.50247849e-05 -3.46008739e-06  2.22044605e-16
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.000

In [29]:
print('norm:', np.linalg.norm(finite_diff_analytical_jac[:3,3,:] - analytical_jac_mj[:3,3,:]))
print('norm:', np.linalg.norm(finite_diff_analytical_jac[:3,3,:] - analytical_jac[:3,3,:]))


norm: 2.1957860504662325
norm: 0.00018818726523442975


In [30]:
# making sure only the first 8 elements are nonzero
assert np.allclose(analytical_jac[:,8:], 0)
assert np.allclose(analytical_jac_mj[:,8:], 0)
assert np.allclose(finite_diff_analytical_jac[:,8:], 0)

In [31]:
# verify if the mujoco is acutally using analytical jacobian for the positional components
assert np.allclose(jacs_mj[:3,:8], analytical_jac[:3,3,:8])
# success! This means that Mujoco is using analytical Jacobian for the position part

In [32]:
for eps in [1e-10, 1e-12, 1e-14, 1e-16]:
    print('eps: ', eps)
    # numerical jacobian of the pose
    numerical_jacobian = naive_finite_diff_jacobian(lambda q: link_pose(q, link_name), q, eps=eps)
    # analytical jacobian of the pose
    analytical_jacobian_mj = get_analytical_jacobian(jacs_mj, pose)
    analytical_jacobian = get_analytical_jacobian(jacs, pose)
    # check the difference
    print('diff: ')
    diff = numerical_jacobian - analytical_jacobian
    print(diff[:,:,:8])
    print('norm: ', np.linalg.norm(diff))

    # check position difference
    print('position diff:')
    print(diff[:3,3,:8])
    print('norm:', np.linalg.norm(diff[:3,3,:8]))
    print('rotation diff:')
    print(diff[:3,:3,:8])
    print('norm:', np.linalg.norm(diff[:3,:3,:8]))

    print('diff_mj: ')
    diff_mj = numerical_jacobian - analytical_jacobian_mj
    print(diff_mj[:,:,:8])
    print('norm_mj: ', np.linalg.norm(diff_mj))
    # check position difference
    print('position diff_mj:')
    print(diff_mj[:3,3,:8])
    print('norm_mj:', np.linalg.norm(diff_mj[:3,3,:8]))
    print('rotation diff_mj:')
    print(diff_mj[:3,:3,:8])
    print('norm_mj:', np.linalg.norm(diff_mj[:3,:3,:8]))

    # additional checks
    diff_mj_2 = numerical_jacobian[:3,3,:] - jacs_mj[:3,:]
    print('diff_mj_2: ', diff_mj_2[:,:8])
    print('norm diff_mj_2: ', np.linalg.norm(diff_mj_2[:,:8]))

eps:  1e-10
diff: 
[[[-2.81292916e-07 -3.56931775e-07  1.67187090e-06 -1.65533734e-06
    1.57036077e-06 -4.69700589e-07 -1.13791527e-06  6.70723673e-07]
  [ 2.16909258e-06  4.06646807e-07 -5.40395850e-09  1.22401185e-06
   -2.53449629e-06  3.66879643e-07  8.56321852e-07  4.65012289e-07]
  [ 1.06186158e-06  2.82120594e-07  1.51954079e-06 -9.50376801e-07
   -8.78263121e-07  1.06119717e-08 -1.09089509e-06  5.55111512e-07]
  [ 7.24424379e-07  6.74595257e-07  2.31335656e-07 -5.14772554e-07
    1.22992102e-07 -2.53706440e-07 -2.32926563e-07 -1.11022302e-16]]

 [[ 1.87456533e-07  1.06687769e-06  4.78387369e-07 -1.04308769e-07
   -1.13922165e-06 -9.54592074e-07 -9.71656886e-07 -3.28909227e-07]
  [-9.48279429e-07 -3.67989675e-07  1.71767681e-06 -1.77988188e-06
   -5.19151409e-07  1.36786732e-06 -3.09672008e-06 -5.51374352e-07]
  [-2.46831370e-06  7.01285395e-07 -6.38487883e-07  5.84012020e-07
   -3.32259701e-08 -1.07525075e-06 -1.20245303e-06 -5.55111512e-07]
  [-1.18286145e-06  1.33377499e-06

In [34]:
# check the newly implemented robot.get_get_link_spatial_jacobian and robot.get_link_analytical_jacobian
robot_spatial_jac = robot.get_link_spatial_jacobian(link_name)
kin_spatial_jac = kin_tree.get_link_spatial_jacobian(link_name, True, False)
assert np.allclose(robot_spatial_jac, kin_spatial_jac)
# verified that the robot implementation now is the same as the textbook
robot_analytical_jac = robot.get_link_analytical_jacobian(link_name)
kin_analytical_jac = get_analytical_jacobian(kin_spatial_jac, pose)
assert np.allclose(robot_analytical_jac, kin_analytical_jac)
# verified that the robot implementation of the analytical jacobian is the same as the textbook